# تمرین عملی فصل ۱۲ ـ پایتون و اکسل

## پاک‌سازی داده، تحلیل و تولید گزارش اکسل

در این تمرین، یک مجموعه‌داده فروش با حدود **۱۰٬۰۰۰ ردیف** را با استفاده از **pandas** وارد و پاک‌سازی می‌کنید و سپس با استفاده از **openpyxl** یک گزارش خلاصه و قالب‌بندی‌شده در Excel ایجاد خواهید کرد.

### فایل مورد استفاده
فایل `ch12_python_excel_practice_large_data.xlsx` را در همان پوشه‌ای قرار دهید که این Notebook را اجرا می‌کنید.

> **نکته آموزشی:** فایل اکسل فقط شامل صورت تمرین و داده‌های خام است. ایمیل‌های نامعتبر و مقادیر گم‌شده عمداً در داده‌ها باقی مانده‌اند تا مراحل پاک‌سازی را خودتان انجام دهید. این Notebook فایل ورودی را بازنویسی نمی‌کند و خروجی جداگانه‌ای با نام `ch12_python_excel_output.xlsx` می‌سازد.

> **نکته اجرایی:** برای جلوگیری از سنگین شدن نمودار Pairplot در مجموعه‌داده بزرگ، از نمونه‌ای تصادفی با حداکثر ۱٬۵۰۰ رکورد برای مصورسازی استفاده می‌شود؛ پاک‌سازی و آمار توصیفی همچنان بر اساس کل داده‌های پاک‌سازی‌شده انجام می‌شود.


## بخش ۱ ـ ورود و بررسی اولیه داده‌ها

داده‌های موجود در شیت **«داده‌های فروش»** را با `pandas` وارد کنید و DataFrame را با نام `sales` ذخیره کنید.

سپس:
1. پنج ردیف نخست را نمایش دهید.
2. تعداد ردیف‌ها و ستون‌ها را بررسی کنید.
3. نوع داده ستون‌ها را مشاهده کنید.


In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.chart import BarChart, Reference
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo

input_path = Path('ch12_python_excel_practice_large_data.xlsx')

if not input_path.exists():
    raise FileNotFoundError(
        f'فایل ورودی پیدا نشد: {input_path.resolve()}\\n'
        'فایل اکسل را در همان پوشه‌ای قرار دهید که Notebook را اجرا می‌کنید.'
    )

sales = pd.read_excel(
    input_path,
    sheet_name='داده‌های فروش'
)

print(f'تعداد رکوردهای خام: {len(sales):,}')
sales.head()


In [ ]:
# ابعاد داده
sales.shape


In [ ]:
# نوع داده ستون‌ها
sales.dtypes


## بخش ۲ ـ استانداردسازی نام ستون‌ها

برای ساده‌تر شدن کار با ستون‌ها، نام آن‌ها را به نام‌های انگلیسی استاندارد تبدیل کنید.

از نگاشت زیر استفاده کنید:

- شناسه تراکنش → `transaction_id`
- تاریخ → `date`
- فروشنده → `salesperson`
- منطقه → `region`
- دسته محصول → `product_category`
- مبلغ فروش → `sales_amount`
- تعداد → `quantity`
- ساعت مطالعه آموزشی → `study_hours`
- امتیاز پیش‌آزمون → `pre_score`
- امتیاز پس‌آزمون → `post_score`
- ایمیل مشتری → `customer_email`

پس از تغییر نام، پنج ردیف نخست را مجدداً بررسی کنید.


In [ ]:
sales = sales.rename(columns={
    'شناسه تراکنش': 'transaction_id',
    'تاریخ': 'date',
    'فروشنده': 'salesperson',
    'منطقه': 'region',
    'دسته محصول': 'product_category',
    'مبلغ فروش': 'sales_amount',
    'تعداد': 'quantity',
    'ساعت مطالعه آموزشی': 'study_hours',
    'امتیاز پیش‌آزمون': 'pre_score',
    'امتیاز پس‌آزمون': 'post_score',
    'ایمیل مشتری': 'customer_email'
})

sales.head()


## بخش ۳ ـ پاکسازی آدرس‌های ایمیل

در مجموعه‌داده چند آدرس ایمیل نامعتبر وجود دارد. با استفاده از **Regular Expression** و متد `str.contains()` فقط رکوردهایی را نگه دارید که آدرس ایمیل معتبر دارند.

سپس تعداد رکوردهای قبل و بعد از پاکسازی را با `shape` مقایسه کنید.

> **نکته آموزشی:** در داده‌های واقعی، اعتبارسنجی الگوهای متنی یکی از جاهایی است که استفاده از Python می‌تواند انعطاف بیشتری نسبت به ابزارهای ساده Excel ایجاد کند.


In [ ]:
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

valid_sales = sales[
    sales['customer_email'].str.contains(email_pattern, na=False, regex=True)
].copy()

print('ابعاد داده اولیه:', sales.shape)
print('ابعاد داده پس از اعتبارسنجی ایمیل:', valid_sales.shape)


## بخش ۴ ـ بررسی و مدیریت مقادیر گم‌شده

ابتدا درصد مقادیر گم‌شده در هر ستون را محاسبه و به ترتیب نزولی مرتب کنید.

سپس رکوردهایی را که در هر یک از ستون‌های ضروری دارای مقدار گم‌شده هستند حذف کنید.

در این تمرین، ستون‌های زیر ضروری در نظر گرفته شده‌اند:

`region`، `product_category` و `study_hours`

در داده‌های تمرینی، مقادیر خالی عمداً در همین ستون‌ها قرار داده شده‌اند. در پروژه‌های واقعی باید پیش از حذف رکوردها، اثر این تصمیم بر تحلیل را نیز بررسی کنید.


In [ ]:
missing_rate = (
    valid_sales.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('درصد مقادیر گم‌شده')
)

missing_rate


In [ ]:
complete_sales = valid_sales.dropna(
    subset=['region', 'product_category', 'study_hours']
).copy()

complete_sales.info()


## بخش ۵ ـ ایجاد رتبه صدکی

برای ستون **امتیاز پس‌آزمون** رتبه صدکی ایجاد کنید و نتیجه را در ستونی با نام `post_pct` قرار دهید.

سپس با استفاده از `describe()` آمار توصیفی این متغیر را بررسی کنید.


In [ ]:
complete_sales['post_pct'] = complete_sales['post_score'].rank(pct=True)

complete_sales['post_pct'].describe()


## بخش ۶ ـ بررسی نهایی داده‌های پاکسازی‌شده

با استفاده از `describe()` یک نمای آماری از متغیرهای عددی ایجاد کنید.

در گزارش خود به‌صورت مفهومی بررسی کنید:
- میانگین مبلغ فروش چقدر است؟
- میانگین امتیاز پیش‌آزمون و پس‌آزمون چه تفاوتی دارند؟
- حداقل و حداکثر مبلغ فروش چیست؟
- رتبه صدکی پس‌آزمون چه دامنه‌ای دارد؟


In [ ]:
complete_sales.describe()


# بخش ۷ ـ ایجاد کارپوشه گزارش با openpyxl

اکنون نقش `pandas` در پاکسازی و تحلیل داده به پایان می‌رسد و از `openpyxl` برای ساخت خروجی Excel استفاده می‌کنیم.

یک کارپوشه جدید ایجاد کنید و شیت اول را با نام **«خلاصه»** قرار دهید.


In [ ]:
wb = Workbook()
ws = wb.active
ws.title = 'خلاصه'


## بخش ۸ ـ ایجاد خلاصه مدیریتی

در شیت **خلاصه**، موارد زیر را درج کنید:

| عنوان | مقدار |
|---|---|
| میانگین امتیاز پیش‌آزمون | میانگین `pre_score` |
| میانگین امتیاز پس‌آزمون | میانگین `post_score` |
| تعداد رکوردهای نهایی | تعداد رکوردهای `complete_sales` |
| میانگین مبلغ فروش | میانگین `sales_amount` |

اعداد اعشاری را تا دو رقم گرد کنید.


In [ ]:
summary = [
    ('میانگین امتیاز پیش‌آزمون', round(complete_sales['pre_score'].mean(), 2)),
    ('میانگین امتیاز پس‌آزمون', round(complete_sales['post_score'].mean(), 2)),
    ('تعداد رکوردهای نهایی', len(complete_sales)),
    ('میانگین مبلغ فروش', round(complete_sales['sales_amount'].mean(), 2))
]

for row_num, (label, value) in enumerate(summary, start=1):
    ws.cell(row=row_num, column=1, value=label)
    ws.cell(row=row_num, column=2, value=value)

ws.column_dimensions['A'].width = 28


## بخش ۹ ـ ایجاد نمودار بومی Excel

بر اساس میانگین امتیاز پیش‌آزمون و پس‌آزمون، یک **نمودار ستونی بومی Excel** ایجاد کنید.

نمودار باید:
- عنوان مناسب داشته باشد؛
- محور افقی شامل نوع امتیاز باشد؛
- محور عمودی مقدار امتیاز را نشان دهد؛
- راهنمای نمودار در صورت غیرضروری بودن حذف شود.

نمودار را از سلول `D1` در شیت «خلاصه» قرار دهید.


In [ ]:
chart = BarChart()

data = Reference(ws, min_col=2, min_row=1, max_row=2)
categories = Reference(ws, min_col=1, min_row=1, max_row=2)

chart.add_data(data)
chart.set_categories(categories)
chart.title = 'مقایسه میانگین امتیازها'
chart.x_axis.title = 'نوع امتیاز'
chart.y_axis.title = 'میانگین امتیاز'
chart.legend = None

ws.add_chart(chart, 'D1')


## بخش ۱۰ ـ ایجاد شیت داده‌های پاکسازی‌شده

یک شیت جدید با نام **«داده‌های پاکسازی‌شده»** ایجاد کنید و DataFrame نهایی `complete_sales` را به آن منتقل کنید.

سپس محدوده داده را به یک **Excel Table** تبدیل کنید.


In [ ]:
ws2 = wb.create_sheet(title='داده‌های پاکسازی‌شده')

for row in dataframe_to_rows(
    complete_sales,
    index=False,
    header=True
):
    ws2.append(row)

table = Table(
    displayName='CleanSalesData',
    ref=f'A1:{get_column_letter(ws2.max_column)}{ws2.max_row}'
)

style = TableStyleInfo(
    name='TableStyleMedium2',
    showRowStripes=True
)

table.tableStyleInfo = style
ws2.add_table(table)


## بخش ۱۱ ـ قالب‌بندی ستون رتبه صدکی

ستون `post_pct` را در شیت «داده‌های پاکسازی‌شده» پیدا کنید و قالب نمایش آن را به درصد با یک رقم اعشار تغییر دهید.

> **نکته آموزشی:** در این مرحله تفاوت نمایه‌گذاری صفرمبنا در Python و شماره‌گذاری ستون‌ها در Excel اهمیت پیدا می‌کند.


In [ ]:
post_pct_loc = complete_sales.columns.get_loc('post_pct') + 1
post_pct_col = get_column_letter(post_pct_loc)

for cell in ws2[post_pct_col]:
    cell.number_format = '0.0%'


## بخش ۱۲ ـ قالب‌بندی شرطی

برای افزایش خوانایی گزارش، قالب‌بندی شرطی را با `PatternFill` پیاده‌سازی کنید:

- رکوردهایی که `post_pct` آن‌ها بیشتر از ۰٫۹ است، با پس‌زمینه سبز مشخص شوند.
- رکوردهایی که `post_pct` آن‌ها بیشتر از ۰٫۷ و حداکثر ۰٫۹ است، با پس‌زمینه زرد مشخص شوند.


In [ ]:
green_fill = PatternFill(
    start_color='B9E8A2',
    end_color='B9E8A2',
    fill_type='solid'
)

yellow_fill = PatternFill(
    start_color='FFF9D4',
    end_color='FFF9D4',
    fill_type='solid'
)

for row in ws2.iter_rows(
    min_row=2,
    min_col=1,
    max_col=len(complete_sales.columns)
):
    post_pct = row[post_pct_loc - 1].value

    if post_pct > 0.9:
        for cell in row:
            cell.fill = green_fill
    elif post_pct > 0.7:
        for cell in row:
            cell.fill = yellow_fill


## بخش ۱۳ ـ تنظیم عرض ستون‌ها

با استفاده از Python، طولانی‌ترین مقدار هر ستون را پیدا کنید و عرض ستون را متناسب با محتوا تنظیم کنید.

هدف این است که کاربر نهایی بدون نیاز به تنظیم دستی عرض ستون‌ها بتواند داده‌ها را به‌راحتی مشاهده کند.


In [ ]:
for column in ws2.columns:
    max_length = 0
    column_letter = column[0].column_letter

    for cell in column:
        try:
            max_length = max(max_length, len(str(cell.value)))
        except Exception:
            pass

    adjusted_width = (max_length + 2) * 1.2
    ws2.column_dimensions[column_letter].width = adjusted_width


# برای جلوگیری از طولانی شدن رسم نمودار روی مجموعه‌داده بزرگ،
# حداکثر ۱٬۵۰۰ ردیف به‌صورت تصادفی برای مصورسازی انتخاب می‌شود.
# تحلیل‌های آماری و پاک‌سازی همچنان از کل complete_sales استفاده می‌کنند.

plot_columns = [
    'sales_amount',
    'quantity',
    'study_hours',
    'pre_score',
    'post_score'
]

plot_data = complete_sales[plot_columns]
plot_sample = plot_data.sample(
    n=min(1500, len(plot_data)),
    random_state=42
)

plot = sns.pairplot(
    plot_sample,
    diag_kind='hist'
)

plot.savefig('sales_pairplot.png', dpi=150, bbox_inches='tight')


In [ ]:
plot = sns.pairplot(
    complete_sales[
        ['sales_amount', 'quantity', 'study_hours', 'pre_score', 'post_score']
    ]
)

plot.savefig('sales_pairplot.png', dpi=150, bbox_inches='tight')


## بخش ۱۵ ـ درج تصویر نمودار پایتونی در Excel

تصویر ذخیره‌شده را با `openpyxl` وارد شیت «خلاصه» کنید و آن را از سلول `A8` قرار دهید.

**نکته مهم:** این نمودار یک تصویر ثابت است و مانند نمودار بومی Excel با تغییر داده‌های کاربرگ به‌صورت خودکار به‌روزرسانی نمی‌شود.


In [ ]:
img = Image('sales_pairplot.png')
ws.add_image(img, 'A8')


# بخش ۱۶ ـ ذخیره خروجی نهایی

کارپوشه نهایی را با نام زیر ذخیره کنید:

`ch12_python_excel_output.xlsx`

در پایان بررسی کنید که فایل شامل:
1. شیت «خلاصه»
2. شیت «داده‌های پاکسازی‌شده»
3. جدول داده‌ها
4. نمودار بومی Excel
5. تصویر نمودار Python
6. قالب‌بندی درصدی و شرطی

باشد.


In [ ]:
output_path = 'ch12_python_excel_output.xlsx'
wb.save(output_path)

print(f'فایل خروجی ذخیره شد: {Path(output_path).resolve()}')
print(f'رکوردهای ورودی: {len(sales):,}')
print(f'رکوردهای نهایی پس از پاک‌سازی: {len(complete_sales):,}')


# جمع‌بندی آموزشی تمرین

در این تمرین یک گردش‌کار کامل و قابل تکرار برای همکاری Python و Excel ایجاد کردید:

**Excel → pandas → پاکسازی و تحلیل → openpyxl → گزارش Excel**

`pandas` برای کار با داده‌های جدولی و عملیات پاکسازی و تحلیل استفاده شد؛ `seaborn` برای مصورسازی تحلیلی و `openpyxl` برای ساخت، قالب‌بندی و تکمیل کارپوشه Excel به کار رفت.

تفاوت مهم دو رویکرد مصورسازی نیز در تمرین مشاهده شد: نمودار بومی Excel قابلیت تعامل و به‌روزرسانی دارد، در حالی که نمودار تولیدشده با Python و ذخیره‌شده به‌صورت تصویر، انعطاف بیشتری در مصورسازی دارد اما در Excel یک تصویر ثابت محسوب می‌شود.
